# Indusense — Sprint 1

Ce notebook produira les cartes d'identité des fichiers CSV placés dans `datas/`.

In [ ]:
import csv
from codecs import BOM_UTF8
from datetime import datetime
from io import StringIO
from pathlib import Path

import pandas as pd

DATA_DIR = Path("datas")

## Lire les fichiers CSV

On conserve les valeurs brutes sous forme de texte pour préserver les identifiants. Les champs vides sont considérés comme manquants. L'encodage est déduit en essayant UTF-8, puis Windows-1252 et ISO-8859-1.

In [ ]:
def lire_csv(chemin: Path) -> tuple[pd.DataFrame, str, str]:
    contenu = chemin.read_bytes()
    encodages = ("utf-8-sig",) if contenu.startswith(BOM_UTF8) else (
        "utf-8", "cp1252", "iso-8859-1"
    )
    for encodage in encodages:
        try:
            texte = contenu.decode(encodage)
        except UnicodeDecodeError:
            continue
        break
    else:
        raise ValueError(f"{chemin}: le fichier ne peut pas être décodé.")

    try:
        separateur = csv.Sniffer().sniff(texte[:8192], delimiters=",;\t").delimiter
    except csv.Error as erreur:
        raise ValueError(f"{chemin}: séparateur CSV indéterminé.") from erreur

    try:
        donnees = pd.read_csv(
            StringIO(texte),
            sep=separateur,
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )
    except (pd.errors.ParserError, pd.errors.EmptyDataError) as erreur:
        raise ValueError(f"{chemin}: lecture du CSV impossible ({erreur}).") from erreur

    return donnees, encodage, separateur

## Inférer les types et calculer les statistiques

Le nom des colonnes guide l'interprétation des identifiants, des indicateurs et des champs de date, d'heure ou de mesure connus. Pour les autres colonnes, on retient le type numérique si toutes les valeurs présentes se convertissent en nombres finis. Les statistiques ignorent les champs vides et ne concernent que les mesures, le score de sévérité, les dates et les heures.

In [ ]:
IDENTIFIANTS = {"incident_id", "machine_id", "operator_badge"}
NUMERIQUES = {
    "severity", "temperature_c", "pressure_bar", "voltage_mean_v",
    "rotation_mean_rpm", "pieces_produced",
}
FORMATS_TEMPORELS = {
    "date": ("DATE", "%Y-%m-%d"),
    "time": ("TIME", "%H:%M"),
    "timestamp": ("DATETIME", "%Y-%m-%d %H:%M:%S"),
}
INDICATEURS = {"0": False, "1": True}


def convertir_numeriques(valeurs: pd.Series) -> pd.Series:
    """Écarte aussi les valeurs infinies des statistiques."""
    nombres = pd.to_numeric(valeurs, errors="coerce")
    return nombres.mask(nombres.isin([float("inf"), float("-inf")]))


def inferer_type(nom: str, valeurs: pd.Series) -> tuple[str, pd.Series]:
    """Renvoie le type et les valeurs non manquantes converties."""
    if nom in IDENTIFIANTS:
        return "IDENTITY", valeurs
    if nom.startswith("type_"):
        return "BOOLEAN", valeurs.map(INDICATEURS)
    if nom in FORMATS_TEMPORELS:
        type_colonne, format_date = FORMATS_TEMPORELS[nom]
        dates = pd.to_datetime(valeurs, format=format_date, errors="coerce")
        return type_colonne, dates

    nombres = convertir_numeriques(valeurs)
    if nom in NUMERIQUES or (not valeurs.empty and nombres.notna().all()):
        return "NUMERIC", nombres
    return "TEXT", valeurs


def calculer_statistiques(
    nom: str, type_colonne: str, valeurs: pd.Series
) -> dict[str, int | float | str]:
    if valeurs.empty:
        return {}
    if type_colonne == "NUMERIC":
        return {
            "min": valeurs.min().item(),
            "max": valeurs.max().item(),
            "mean": valeurs.mean().item(),
            "median": valeurs.median().item(),
        }
    if nom in FORMATS_TEMPORELS:
        _, format_date = FORMATS_TEMPORELS[nom]
        return {
            "min": valeurs.min().strftime(format_date),
            "max": valeurs.max().strftime(format_date),
        }
    return {}

## Décrire les deux fichiers

Chaque description regroupe les métadonnées, le type des colonnes et les statistiques. Les horaires sont ceux de la machine qui exécute le notebook. La section suivante utilise ces descriptions pour générer les cartes Markdown.

In [ ]:
def decrire_csv(chemin: Path) -> dict[str, object]:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    taille = source.stat().st_size
    donnees, encodage, separateur = lire_csv(source)
    colonnes = []

    for nom, serie in donnees.items():
        # Un champ composé seulement d'espaces compte aussi comme manquant.
        valeurs = serie[serie.notna() & serie.str.strip().ne("")]
        type_colonne, converties = inferer_type(nom, valeurs)
        invalides = int(converties.isna().sum())
        if invalides:
            raise ValueError(
                f"{source.name}, colonne {nom}: {invalides} valeur(s) "
                f"invalide(s) pour le type {type_colonne}."
            )
        colonnes.append({
            "nom": nom,
            "type": type_colonne,
            "valeurs_manquantes": len(serie) - len(valeurs),
            "statistiques": calculer_statistiques(nom, type_colonne, converties),
        })

    lignes, nombre_colonnes = donnees.shape
    return {
        "source": str(source),
        "taille_octets": taille,
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "separateur": separateur,
        "lignes": lignes,
        "nombre_colonnes": nombre_colonnes,
        "colonnes": colonnes,
    }

In [ ]:
FICHIERS_CSV = ("releves_incidents.csv", "telemetry.csv")
descriptions = {nom: decrire_csv(DATA_DIR / nom) for nom in FICHIERS_CSV}

pd.DataFrame([
    {
        "Fichier": nom,
        "Encodage": description["encodage"],
        "Séparateur": repr(description["separateur"]),
        "Lignes": description["lignes"],
        "Colonnes": description["nombre_colonnes"],
    }
    for nom, description in descriptions.items()
])

## Générer les cartes d'identité Markdown

Chaque carte prend le nom du fichier source et l'heure de début de son analyse. Les rapports sont enregistrés dans le dossier daté sous `artifacts/ingestion/identities/`.

In [ ]:
IDENTITIES_DIR = Path("artifacts/ingestion/identities")


def formater_taille(octets: int) -> str:
    taille = float(octets)
    for unite in ("octets", "Ko", "Mo", "Go", "To"):
        if taille < 1_000 or unite == "To":
            return f"{taille:.2f} {unite}"
        taille /= 1_000


def rendre_carte_markdown(description: dict[str, object]) -> str:
    """Met en forme une description déjà calculée."""
    source = Path(description["source"])
    debut = description["debut"]
    fin = description["fin"]
    lignes = [
        f"# Carte d'identité — {source.name}",
        "",
        f"- Source: `{source}`",
        f"- Filsize: {formater_taille(description['taille_octets'])}",
        f"- ProcessBeginAt: {debut:%H:%M:%S}",
        f"- ProcessEndAt: {fin:%H:%M:%S}",
        f"- Charset: {description['encodage']}",
        f"- Rows Number: {description['lignes']}",
        f"- Cols Number: {description['nombre_colonnes']}",
        "",
        "## Column list",
        "",
        "| Colname | Inferred Type |",
        "| --- | --- |",
    ]
    statistiques = []
    for colonne in description["colonnes"]:
        nom = colonne["nom"].replace("\\", "\\\\").replace("|", "\\|")
        type_colonne = colonne["type"]
        lignes.append(f"| {nom} | {type_colonne} |")
        if type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            valeurs = colonne["statistiques"]
            cellules = [str(valeurs.get(cle, "—")) for cle in ("min", "max", "mean", "median")]
            statistiques.append("| " + " | ".join([nom, *cellules]) + " |")

    if statistiques:
        lignes.extend([
            "",
            "## Statistiques",
            "",
            "| Colname | Min value | Max value | Mean value | Median value |",
            "| --- | --- | --- | --- | --- |",
            *statistiques,
        ])
    return "\n".join(lignes) + "\n"


def ecrire_carte_markdown(description: dict[str, object], destination: Path) -> Path:
    """Enregistre une carte sans écraser un rapport existant."""
    debut = description["debut"]
    dossier = destination / f"{debut:%Y%m%d}"
    nom_source = Path(description["source"]).stem
    chemin = dossier / f"{nom_source}_{debut:%H%M%S}.md"
    contenu = rendre_carte_markdown(description)
    dossier.mkdir(parents=True, exist_ok=True)
    with chemin.open("x", encoding="utf-8", newline="\n") as fichier:
        fichier.write(contenu)
    return chemin

In [ ]:
cartes_markdown = {
    nom: ecrire_carte_markdown(description, IDENTITIES_DIR)
    for nom, description in descriptions.items()
}

for chemin in cartes_markdown.values():
    print(chemin.as_posix())